<h1 style='color:#A23B72'>Optimización Regional: Comparativa Global vs Regional (500 Curvas)</h1>

<p style='color:#b0b0b0'>Comparamos desempeño global vs segmentación regional en las 500 curvas del dataset.</p>

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import find_peaks
from scipy.optimize import curve_fit

plt.rcParams.update({'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False})

ROOT = Path.cwd().parent
DATA_T = ROOT / 'datos' / 'target'
DATA_P = ROOT / 'datos' / 'pixel_curves'
RESULTADOS_DIR = ROOT / 'metodo_gaussiano' / 'resultados'

# Cargar resultados globales
df_ganadores = pd.read_csv(RESULTADOS_DIR / 'completo_configuraciones_ganadoras.csv')

print(f'✓ Datos cargados: {len(df_ganadores)} curvas')

## 2. Funciones Base

In [ ]:
def leer_target(cid):
    return pd.read_csv(DATA_T / f'curve_{cid:04d}.txt', header=None, names=['x', 'y']).sort_values('x').reset_index(drop=True)

def leer_pixel_real(cid, escala=25):
    d = pd.read_csv(DATA_P / f'curve_{cid:04d}_X{escala}.txt', sep=' ', header=None, names=['x', 'y'])
    f = escala / 10.0
    d['x'] = d['x'] / f
    d['y'] = d['y'] / f
    return d.sort_values('x').reset_index(drop=True)

def calcular_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - y_true.mean())**2)
    return 1 - (ss_res / ss_tot) if ss_tot > 0 else 0

def suma_gaussianas(x, *params):
    n_campanas = (len(params) - 1) // 3
    c = params[-1]
    y = np.full_like(x, c, dtype=float)
    for i in range(n_campanas):
        A, mu, sig = params[3*i], params[3*i+1], params[3*i+2]
        y += A * np.exp(-(x - mu)**2 / (2 * sig**2 + 1e-9))
    return y

def inicializar_desde_picos(x, y, n_campanas, metodo='altura'):
    y_centered = y - y.mean()
    if metodo == 'altura':
        pks, _ = find_peaks(y_centered, height=y_centered.mean())
        if len(pks) == 0:
            pks = np.array([np.argmax(y_centered)])
        idx_top = np.argsort(y_centered[pks])[-n_campanas:]
        picos_idx = pks[idx_top]
    elif metodo == 'uniforme':
        indices = np.linspace(0, len(x) - 1, n_campanas, dtype=int)
        picos_idx = indices
    else:
        picos_idx = np.linspace(0, len(x) - 1, n_campanas, dtype=int)
    
    p0 = []
    for idx in sorted(picos_idx):
        A = float(y[idx] - y.mean())
        mu = float(x[idx])
        sigma = float((x.max() - x.min()) / (2 * n_campanas + 1))
        p0 += [A, mu, sigma]
    p0.append(float(y.mean()))
    return np.array(p0)

print('Funciones base definidas')

## 3. Segmentación y Optimización Regional

In [ ]:
def segmentar_uniforme(x, y, n_regiones=3):
    """Divide en n_regiones uniformes."""
    x_min, x_max = x.min(), x.max()
    ancho = (x_max - x_min) / n_regiones
    
    regiones = []
    for i in range(n_regiones):
        inicio = x_min + i * ancho
        fin = x_min + (i + 1) * ancho
        mask = (x >= inicio) & (x <= fin)
        if mask.sum() > 0:
            regiones.append((x[mask], y[mask]))
    return regiones

def optimizar_region(x_seg, y_seg, n_campanas=2):
    """Optimiza una región individual."""
    n = max(1, min(n_campanas, 5))
    p0 = inicializar_desde_picos(x_seg, y_seg, n, metodo='uniforme')
    
    try:
        popt, _ = curve_fit(suma_gaussianas, x_seg, y_seg, p0=p0, maxfev=2000)
        y_pred = suma_gaussianas(x_seg, *popt)
        r2 = calcular_r2(y_seg, y_pred)
        return y_pred, r2
    except:
        return y_seg, 0.0

print('Funciones de optimización regional definidas')

## 4. Comparativa Global vs Regional para 500 Curvas

In [ ]:
print("Comparando desempeño global vs regional en 500 curvas...\n")

resultados_comparativa = []

for idx, row in df_ganadores.iterrows():
    if (idx + 1) % 50 == 0:
        print(f"Procesadas {idx + 1}/500 curvas")
    
    cid = int(row['curva_id'])
    tgt = leer_target(cid)
    pix = leer_pixel_real(cid, 25)
    
    x_tgt = tgt['x'].values
    y_tgt = tgt['y'].values
    
    # R² global (del resultado de optimización)
    r2_global = row['r2']
    
    # Optimización regional (3 regiones)
    x_pix = pix['x'].values
    y_pix = pix['y'].values
    
    regiones = segmentar_uniforme(x_pix, y_pix, n_regiones=3)
    
    r2_regional_promedio = 0
    for reg_idx, (x_seg, y_seg) in enumerate(regiones):
        n_camp = max(1, int(row['n_campanas'] / 3) + 1)  # Distribuir campanas
        _, r2_seg = optimizar_region(x_seg, y_seg, n_campanas=n_camp)
        r2_regional_promedio += r2_seg
    
    r2_regional_promedio /= len(regiones)
    
    resultados_comparativa.append({
        'curva_id': cid,
        'r2_global': r2_global,
        'r2_regional': r2_regional_promedio,
        'mejora': r2_regional_promedio - r2_global,
        'mejor_metodo': 'regional' if r2_regional_promedio > r2_global else 'global'
    })

df_comparativa = pd.DataFrame(resultados_comparativa)
print(f"\n✓ Comparativa completada")

## 5. Análisis de Resultados

In [ ]:
print("\n" + "="*70)
print("COMPARATIVA GLOBAL VS REGIONAL (500 CURVAS)")
print("="*70)

print(f"\nR² Global:")
print(df_comparativa['r2_global'].describe().round(4))

print(f"\nR² Regional:")
print(df_comparativa['r2_regional'].describe().round(4))

print(f"\nMejora (Regional - Global):")
print(df_comparativa['mejora'].describe().round(4))

mejoras = (df_comparativa['r2_regional'] > df_comparativa['r2_global']).sum()
print(f"\nCurvas donde regional > global: {mejoras}/500 ({mejoras/500*100:.1f}%)")

mejora_promedio = df_comparativa['mejora'].mean()
print(f"Mejora promedio: {mejora_promedio:.6f}")

## 6. Visualización

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_facecolor('white')

# Scatterplot: Global vs Regional
axes[0, 0].scatter(df_comparativa['r2_global'], df_comparativa['r2_regional'], 
                   alpha=0.5, s=30, color='#A23B72')
lims = [min(df_comparativa['r2_global'].min(), df_comparativa['r2_regional'].min()),
        max(df_comparativa['r2_global'].max(), df_comparativa['r2_regional'].max())]
axes[0, 0].plot(lims, lims, 'k--', alpha=0.3, lw=1)
axes[0, 0].set_xlabel('R² Global')
axes[0, 0].set_ylabel('R² Regional')
axes[0, 0].set_title('Global vs Regional R²')
axes[0, 0].grid(alpha=0.3)

# Distribución de mejora
axes[0, 1].hist(df_comparativa['mejora'], bins=50, color='#5BC0EB', alpha=0.7)
axes[0, 1].axvline(0, color='red', linestyle='--', lw=2)
axes[0, 1].set_xlabel('Mejora (Regional - Global)')
axes[0, 1].set_ylabel('Frecuencia')
axes[0, 1].set_title('Distribución de Mejoras')
axes[0, 1].grid(alpha=0.3)

# Boxplot comparativo
axes[1, 0].boxplot([df_comparativa['r2_global'], df_comparativa['r2_regional']], 
                    labels=['Global', 'Regional'])
axes[1, 0].set_ylabel('R²')
axes[1, 0].set_title('Distribución de R² Comparativa')
axes[1, 0].grid(alpha=0.3)

# Conteo: mejor método
metodos = df_comparativa['mejor_metodo'].value_counts()
axes[1, 1].bar(metodos.index, metodos.values, color=['#A23B72', '#5BC0EB'])
axes[1, 1].set_ylabel('Cantidad de curvas')
axes[1, 1].set_title('Cuál es mejor: Global vs Regional')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Conclusiones

In [ ]:
print("\n" + "="*70)
print("CONCLUSIONES")
print("="*70)

if mejoras / 500 > 0.5:
    print(f"\n✓ RECOMENDACIÓN: Usar optimización REGIONAL")
    print(f"  - {mejoras}/500 curvas mejoran con regional ({mejoras/500*100:.1f}%)")
    print(f"  - Mejora promedio: +{mejora_promedio:.6f} en R²")
else:
    print(f"\n✓ RECOMENDACIÓN: Usar optimización GLOBAL")
    print(f"  - Solo {mejoras}/500 curvas mejoran con regional ({mejoras/500*100:.1f}%)")
    print(f"  - Mejor mantener simplicidad del método global")

# Guardar resultados
df_comparativa.to_csv(RESULTADOS_DIR / 'comparativa_global_regional.csv', index=False)
print(f"\n✓ Resultados guardados en {RESULTADOS_DIR}/comparativa_global_regional.csv")